In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, window, sum as _sum
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

# Start Spark session with Kafka package
spark = SparkSession.builder \
    .appName("KafkaWindowParquetDemo") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Define schema of Kafka messages
trip_schema = StructType([
    StructField("pickup_datetime", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("passenger_count", IntegerType(), True)
])

# Read streaming data from Kafka
df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "trips") \
    .option("startingOffsets", "earliest") \
    .load()

# Parse JSON messages
trips = df.selectExpr("CAST(value AS STRING) as json") \
          .select(from_json(col("json"), trip_schema).alias("data")) \
          .select("data.*")

# Cast timestamp
trips = trips.withColumn("pickup_datetime", col("pickup_datetime").cast(TimestampType()))

# Window + Watermark
agg = trips.withWatermark("pickup_datetime", "0 minutes") \
           .groupBy(window(col("pickup_datetime"), "1 minute"),col("PULocationID")).agg(_sum(col("passenger_count")).alias('total_passenger'))

# Write to Parquet (append mode + checkpoint)
query = agg.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("path", "/home/jovyan/work/output_parquet") \
    .option("checkpointLocation", "/home/jovyan/work/checkpoint") \
    .start()

query.awaitTermination()